# Week 12B: From a notebook to a public website

Turn the DuckDB analysis into an HTML report, use ChatGPT to help build a small website, preview it on localhost, and publish with GitHub Pages.

<a href="lecture-12B.ipynb" download>Download these notes as a notebook</a> · [Previous: DuckDB Spatial](lecture-12A.html)

**Deliverable:** a GitHub Pages link containing a rendered notebook and a simple Census story page. No new Census download is needed; use the small exports from Part A.

## 1. Understand the workflow

| Stage | You work with | What it does |
| --- | --- | --- |
| Analysis | Python + DuckDB + `.ipynb` | Runs queries and makes exports |
| Report | Quarto + saved notebook outputs | Produces HTML and supporting assets |
| Website | HTML + CSS + JavaScript | Presents the exported results |
| Local preview | A local HTTP server | Lets your browser load the site on your computer |
| Version control | Git + a GitHub repository | Records and shares revisions |
| Hosting | GitHub Pages | Serves the generated static files to visitors |

Git and GitHub are different: Git is the version-control tool, while GitHub stores repositories online. A repository URL shows source files; a GitHub Pages URL shows the website.

GitHub Pages does **not** run a Python kernel, execute DuckDB SQL on your computer, or host a Python API. Exported images/tables and browser JavaScript work; Python callbacks need a separate application server. Keep the `.duckdb` file local for this lab.

## 2. Check tools and finish the analysis

Run Part A, save the notebook with its outputs, and keep:

```text
outputs/
├── category_counts.json
├── category_counts.csv
├── district_counts.csv
├── district_sample_map.png
└── district_sample.geojson
```

For a working comparison, download [the standalone website starter](downloads/starter-website.zip) or [the complete Quarto publishing starter](downloads/quarto-site.zip). Extract a ZIP into a **new project folder outside the course Git repository**. These small starters include precomputed exports, so they can also be used before finishing Part A.

Install [Quarto](https://quarto.org/docs/get-started/) for your operating system and [Git](https://git-scm.com/downloads). Quarto is a command-line publishing tool, not a Python import. Open a new terminal after installation:

```bash
quarto --version
git --version
conda activate census-web
quarto check jupyter
```

[Quarto's getting-started guide](https://quarto.org/docs/get-started/) explains the installation options.

## 3. Render a notebook with Quarto

Quarto combines Markdown, source code, saved tables, and plots into a readable report. From the folder containing your saved analysis notebook:

```bash
quarto render lecture-12A.ipynb --to html
```

Open `lecture-12A.html` in a browser. Keep its supporting `lecture-12A_files/` folder with it. By default, Quarto uses saved outputs in an `.ipynb`; save the notebook after a successful Run All. Use `--execute` only when you intend to rerun the Python analysis in its working folder and environment.

You can add a Markdown cell with a question, explanation, or caption and render again. Good reports explain **why** a query is useful, not just its syntax. Remove failed outputs before sharing. [Quarto's Python/Jupyter guide](https://quarto.org/docs/computations/python.html).

**Try it:** add a short explanation of the spatial boundary rule, then render again.

## 4. Build a small Quarto website

A Quarto project adds navigation and multiple pages. Download and extract [quarto-site.zip](downloads/quarto-site.zip), or create this structure yourself:

```text
quarto-site/
├── _quarto.yml
├── index.qmd
├── report.ipynb
├── assets/                  # small analysis exports
├── website/                 # plain HTML/CSS/JS story page
└── .gitignore
```

`report.ipynb` in the starter is an executed, small notebook showing the exported table and map. It demonstrates the same render-and-host process without rerunning the full database import. You may replace it with your saved analysis notebook, keeping any linked files alongside it.

The project's `_quarto.yml` is:

```yaml
project:
  type: website
  output-dir: docs
  render:
    - index.qmd
    - report.ipynb
  resources:
    - assets/**
    - website/**
website:
  title: "Census database story"
  navbar:
    left:
      - href: index.qmd
        text: Home
      - href: report.ipynb
        text: Notebook report
      - href: website/index.html
        text: Story website
format:
  html:
    toc: true
execute:
  enabled: false
```

The project renders to `docs/`, the folder we will publish. `execute: enabled: false` makes this starter use the notebook's saved results. The file `index.qmd` is the homepage source; `.qmd` is Markdown with Quarto options. The plain website is copied as a resource. [Quarto website guide](https://quarto.org/docs/websites/).

### Preview and render the Quarto project

In a terminal opened **inside `quarto-site`**:

```bash
quarto preview
```

Use the URL Quarto prints. Make an edit and watch the preview update. Press **Ctrl+C** to stop the preview.

Create the files that will actually be published:

```bash
quarto render
python -c "from pathlib import Path; Path('docs/.nojekyll').touch()"
python -m http.server 8001 --bind 127.0.0.1 --directory docs
```

Visit [http://localhost:8001](http://localhost:8001), test all three navigation links, then press Ctrl+C. `.nojekyll` tells GitHub Pages to serve the rendered files directly. Publish the generated `docs` folder, not just the source `.ipynb`.

## 5. Use ChatGPT to create a website

Here, **vibe coding** means describing a goal, reviewing the generated code and page, testing it, and iterating. You remain responsible for data labels and conclusions. Ask for a small working page first, then request one improvement at a time.

In ChatGPT, provide the exported files when file attachment is available, or paste the small JSON and explain the expected filenames. Ask for complete code files you can save locally. This workflow does not require writing OpenAI API calls or obtaining an API key. For local file editing through a coding agent, use its connected project tools; ordinary chat code blocks do not automatically create files on your computer.

A useful prompt supplies the **goal, audience, inputs, constraints, output files, and acceptance checks**. See [OpenAI's prompting guidance](https://learn.chatgpt.com/docs/prompting).

[Download the full example prompt](examples/website-prompt.txt). Paste it into ChatGPT with your exports:

```text
Create a responsive Census class-project website using my exported
category_counts.json, district_counts.csv, and district_sample_map.png.

Return complete index.html, styles.css, and app.js files, using assets/
for data and the map. Use relative paths, semantic HTML, readable labels,
and a phone-friendly layout. No framework, build step, API key, or CDN.

Show the actual exported counts in a labeled bar chart and display the
map with alt text. Explain that these are synthetic historical Census
teaching dots and 2015 districts, not population estimates or current
demographics. Do not invent counts or claims.

Include methods, source limitations, a data-load error message, and
instructions for previewing with python -m http.server 8000.
```

This is a prompt template for students to adapt, not a promise that every generated response will be correct.

### Save and test the generated files

Create a separate folder for the plain website:

```text
starter-website/
├── index.html
├── styles.css
├── app.js
└── assets/
    ├── category_counts.json
    ├── district_counts.csv
    └── district_sample_map.png
```

Save the returned code without the surrounding Markdown code fences. Put the actual Part A exports in `assets/`. Use the [working starter ZIP](downloads/starter-website.zip) to compare file structure and behavior.

In a terminal opened inside that folder:

```bash
python -m http.server 8000 --bind 127.0.0.1
```

Visit [http://localhost:8000](http://localhost:8000). `localhost` refers to your own computer; classmates cannot use your localhost URL. Stop the server with Ctrl+C. Double-clicking `index.html` can block JavaScript `fetch()` requests, which is why we use HTTP for previewing. [Python HTTP server documentation](https://docs.python.org/3/library/http.server.html).

For inclusion in the Quarto project, copy the **contents** of this folder into `quarto-site/website/`, then rerender. The downloaded Quarto starter already has this structure.

### Iterate using evidence

Check the page before asking for more code:

1. Does the sum of the bars match the exported sample-dot total?
2. Does the map load, and does the caption match what it measures?
3. Can you read the page at phone width and use controls with a keyboard?
4. Are source links and all relative paths working?

Example follow-up prompt:

```text
The page opens at localhost:8000 but the chart is empty. The browser's
console shows a 404 for category_counts.json. My file is actually in
assets/category_counts.json. Explain the cause, then change only the
relative data path and preserve the existing counts and captions.
```

Browser developer tools show errors in **Console** and missing files in **Network**. Copy the exact error rather than saying only that the website does not work.

## 6. Turn a repeated workflow into a skill

A prompt describes one request. A **skill** packages reusable instructions for a workflow. A skill is a folder with a `SKILL.md` file containing a name, description, and instructions; optional resources can accompany it. Skills support depends on the ChatGPT or coding-agent environment you use. If your interface does not support skills, paste the instructions as a reusable prompt instead. [Official skill guide](https://learn.chatgpt.com/docs/build-skills).

[Download the example skill](downloads/census-story-site-skill.zip) or [read its SKILL.md](examples/skills/census-story-site/SKILL.md). It is a teaching example, not installed automatically. In a Codex project, place it at:

```text
.agents/skills/census-story-site/SKILL.md
```

Its front matter begins:

```yaml
---
name: census-story-site
description: Build or revise a static class website from supplied Census teaching-analysis exports.
---
```

The body tells the agent to read the supplied exports, preserve the sample-data limitations, use relative asset paths, test the page, and avoid inventing numbers. Keep the skill specific to its workflow rather than filling it with general advice.

When using Codex, explicitly request `$census-story-site` with your project goal and exported assets. In ChatGPT environments that support skills, use their skill selector. A skill provides instructions; it does not grant account access or automatically publish a site. [Skill creation and use](https://learn.chatgpt.com/docs/build-skills).

## 7. Put the project in Git and GitHub

Use the separate `quarto-site` project folder, not the full course repository. On GitHub, create a **new empty public repository**, for example `census-story`. For this example, do not initialize it with a README or other files, because you already have a local project.

After rendering and checking `docs/`, run these commands in `quarto-site`. Replace `YOUR-NAME` with your GitHub username:

```bash
git init
git branch -M main
git status
git add _quarto.yml index.qmd report.ipynb assets website docs .gitignore
git commit -m "Add Census notebook report and story website"
git remote add origin https://github.com/YOUR-NAME/census-story.git
git push -u origin main
```

Authenticate using GitHub's supported sign-in flow or GitHub Desktop. Do not put a password or access token into source files. If Git requests your identity, configure your chosen commit name and email before committing. The starter `.gitignore` excludes local database files and notebook checkpoints but **does not ignore `docs/`**, because we are publishing that folder.

GitHub Desktop is an alternative: add the separate local repository, review the changed files, commit, and publish the repository. The key idea is to include the rendered files and all their assets. See [GitHub's repository quickstart](https://docs.github.com/en/repositories/creating-and-managing-repositories/quickstart-for-repositories).

## 8. Publish the rendered notebook with GitHub Pages

On your repository:

1. Open **Settings → Pages**.
2. Under **Build and deployment**, choose **Deploy from a branch**.
3. Choose branch **main** and folder **/docs**, then save.
4. Wait for the Pages deployment to complete and open the site link shown there.

For a project repository, the URL normally has this shape:

```text
https://YOUR-NAME.github.io/census-story/
```

Test the homepage, **Notebook report**, and **Story website** on the published site. Relative paths keep assets working under `/census-story/`; avoid paths beginning with `/assets/`, which point at the account's domain root.

This branch-based method uses files already rendered on your machine. GitHub Actions is another option, but is not needed for this lab. GitHub documents the [branch and folder publishing settings](https://docs.github.com/en/pages/getting-started-with-github-pages/configuring-a-publishing-source-for-your-github-pages-site); Quarto provides a [GitHub Pages publishing guide](https://quarto.org/docs/publishing/github-pages.html).

### Update your site

Edit a source page or rerun and save the report notebook as appropriate. Copy refreshed exports into the website assets, then:

```bash
quarto render
python -c "from pathlib import Path; Path('docs/.nojekyll').touch()"
git status
git add _quarto.yml index.qmd report.ipynb assets website docs
git commit -m "Update the Census analysis and website"
git push
```

Refresh the published URL after deployment finishes. Editing a source notebook alone does not refresh committed HTML; rerender and push the new `docs/` files too.

## Troubleshooting

| Symptom | First check |
| --- | --- |
| `quarto` is not found | Install the Quarto CLI and reopen the terminal |
| Report has old plots | Run and save the notebook, then rerender |
| Chart is empty locally | Use HTTP, check Console/Network, and verify the relative JSON path |
| Git push is rejected | Confirm this is the intended repository and review the remote history; do not force push to bypass it |
| Pages gives 404 | Check the deployment status, `main` + `/docs`, and `docs/index.html` |
| Styles/images missing online | Commit all assets and check path spelling/case; test under the repository URL |
| Python controls do not work on Pages | Static hosting does not run Python callbacks; export results or use browser-side interactions |

## In-class exercise

Create a site that explains one spatial query from Part A. Include the rendered notebook, a query-result visualization, a source/limitations paragraph, and a link to your GitHub repository. Submit both the repository URL and the Pages URL, and explain how they differ.